# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")


## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# Discover available record sets and their @ids
record_sets = dataset.record_sets
print(f"Number of record sets: {len(record_sets)}\n")
for rs in record_sets:
    print(f"RecordSet name: {rs.name}")
    print(f"  @id: {rs.id}")
    print(f"  Description: {getattr(rs, 'description', 'No description')}\n")
    print("  Fields:")
    for field in rs.fields:
        print(f"    - {field.name} (@id: {field.id}, type: {field.data_type})")
    print('-' * 60)

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Extract data from each record set
rs_ids = [rs.id for rs in dataset.record_sets]
dataframes = {}

for rs_id in rs_ids:
    records = list(dataset.records(record_set=rs_id))
    dataframes[rs_id] = pd.DataFrame(records)

if rs_ids:
    print(f"Fields in primary RecordSet ({rs_ids[0]}):")
    print(dataframes[rs_ids[0]].columns.tolist())
    dataframes[rs_ids[0]].head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section should include operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
import numpy as np

# Choose a RecordSet and numeric field for EDA automatically (based on list of columns/types found before)
primary_record_set = rs_ids[0] if rs_ids else None

# Attempt to automatically find a numeric field
numeric_field = None
group_field = None
if primary_record_set:
    columns = dataframes[primary_record_set].columns.tolist()
    numeric_candidates = []
    # Try to infer a numeric field name commonly found in proteomics
    for col in columns:
        if any(token in col.lower() for token in ['abund', 'count', 'mw', 'mass', 'weight', 'score', 'coverage', 'peptide']):
            # Only keep those columns with likely numeric content
            numeric_candidates.append(col)
    if numeric_candidates:
        # Pick the first candidate
        numeric_field = numeric_candidates[0]

    # Try to find a field to group by (e.g. 'description', 'sample', 'modification')
    for col in columns:
        if any(token in col.lower() for token in ['description', 'modification', 'sample', 'accession', 'protein', 'group']):
            group_field = col
            break

if numeric_field and numeric_field in dataframes[primary_record_set].columns:
    print(f"Numeric field selected for EDA: {numeric_field}\n")
    # Attempt to coerce to numeric, errors will turn values NaN
    df = dataframes[primary_record_set]
    df[numeric_field] = pd.to_numeric(df[numeric_field], errors='coerce')

    # Drop rows with missing numeric values
    df_filtered = df.dropna(subset=[numeric_field]).copy()

    # Example: Filter for values greater than threshold (use median if no prior info)
    threshold = df_filtered[numeric_field].median()
    filtered_df = df_filtered[df_filtered[numeric_field] > threshold]
    print(f"Filtered records with {numeric_field} > {threshold} (median):\n")
    print(filtered_df.head())

    # Normalize the numeric field (z-score)
    filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"\nNormalized {numeric_field} for filtered records:")
    print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    if group_field and group_field in df_filtered.columns:
        # Only group by categorical fields
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
        print(f"\nGrouped data by {group_field} (showing mean {numeric_field}):")
        print(grouped_df.head())
else:
    print("No suitable numeric field found for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field and numeric_field in dataframes[primary_record_set].columns:
    plt.figure(figsize=(8,4))
    sns.histplot(dataframes[primary_record_set][numeric_field].dropna(), bins=30, kde=True)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel('Frequency')
    plt.show()
    
    # Visualize group means if grouping available
    if group_field and group_field in dataframes[primary_record_set].columns:
        grouped_vals = dataframes[primary_record_set].groupby(group_field)[numeric_field].mean().reset_index()
        plt.figure(figsize=(12,4))
        sns.barplot(data=grouped_vals, x=group_field, y=numeric_field, color='skyblue')
        plt.title(f"Mean {numeric_field} by {group_field}")
        plt.xlabel(group_field)
        plt.ylabel(f"Mean {numeric_field}")
        plt.xticks(rotation=60, ha='right')
        plt.tight_layout()
        plt.show()
else:
    print("No numeric field available for visualization.")

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

* In this notebook, we loaded and explored the FAIR^2-compliant dataset "Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells" using the `mlcroissant` library.
* We discovered available record sets, their fields and types using the Croissant schema's `@id` references, ensuring reproducibility.
* We demonstrated how to dynamically extract all record sets into pandas DataFrames, select fields for numeric analysis, filter, normalize, and group the data by categorical fields when possible.
* Finally, we visualized the selected numeric field's distribution and, if available, its mean by group, illustrating basic EDA with Croissant datasets.

**Next steps:** Depending on your research objectives, you may perform advanced analyses, apply statistical testing, or engineer features for machine learning using this structured approach.